# Rekurrenta neurala nätverk

I den tidigare modulen täckte vi rika semantiska representationer av text. Arkitekturen vi har använt fångar den samlade meningen av ord i en mening, men den tar inte hänsyn till **ordningen** på orden, eftersom aggregeringsoperationen som följer efter inbäddningarna tar bort denna information från den ursprungliga texten. Eftersom dessa modeller inte kan representera ordningsföljd kan de inte lösa mer komplexa eller tvetydiga uppgifter som textgenerering eller frågesvar.

För att fånga meningen i en textsekvens använder vi en neural nätverksarkitektur som kallas **rekurrentt neuralt nätverk**, eller RNN. När vi använder ett RNN skickar vi vår mening genom nätverket ett token i taget, och nätverket producerar ett visst **tillstånd**, som vi sedan skickar in i nätverket igen tillsammans med nästa token.

![Bild som visar ett exempel på rekurrent nätverksgenerering.](../../../../../translated_images/sv/rnn.27f5c29c53d727b5.webp)

Givet inmatningssekvensen av tokens $X_0,\dots,X_n$, skapar RNN en sekvens av neurala nätverksblock och tränar denna sekvens från början till slut med backpropagation. Varje nätverksblock tar ett par $(X_i,S_i)$ som indata och producerar $S_{i+1}$ som resultat. Det slutgiltiga tillståndet $S_n$ eller utdata $Y_n$ går in i en linjär klassificerare för att producera resultatet. Alla nätverksblock delar samma vikter och tränas från början till slut med en backpropagation-omgång.

> Figuren ovan visar rekurrenta neurala nätverket i utfälld form (till vänster) och i en mer kompakt rekurrent representation (till höger). Det är viktigt att inse att alla RNN-celler har samma **delbara vikter**.

Eftersom tillståndsvektorer $S_0,\dots,S_n$ skickas genom nätverket kan RNN lära sig sekventiella beroenden mellan ord. Till exempel, när ordet *inte* uppträder någonstans i sekvensen kan det lära sig att förneka vissa element inom tillståndsvektorn.

Inuti innehåller varje RNN-cell två viktmatriser: $W_H$ och $W_I$, samt bias $b$. Vid varje RNN-steg, givet indata $X_i$ och inmatningstillstånd $S_i$, beräknas utgångstillståndet som $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, där $f$ är en aktiveringsfunktion (ofta $\tanh$).

> För problem som textgenerering (som vi kommer att täcka i nästa enhet) eller maskinöversättning vill vi också få något utdata vid varje RNN-steg. I det här fallet finns en annan matris $W_O$, och utdata beräknas som $Y_i=f(W_O\times S_i+b_O)$.

Låt oss se hur rekurrenta neurala nätverk kan hjälpa oss att klassificera vår nyhetsdatamängd.

> För sandbox-miljön måste vi köra följande cell för att säkerställa att det nödvändiga biblioteket är installerat och att data är förladdad. Om du kör lokalt kan du hoppa över följande cell.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

När du tränar stora modeller kan GPU-minnesallokering bli ett problem. Vi kan också behöva experimentera med olika minibatch-storlekar, så att datan får plats i vårt GPU-minne, men träningen ändå är tillräckligt snabb. Om du kör den här koden på din egen GPU-maskin kan du experimentera med att justera minibatch-storleken för att påskynda träningen.

> **Notera**: Vissa versioner av NVidia-drivrutiner är kända för att inte frigöra minnet efter att ha tränat modellen. Vi kör flera exempel i denna anteckningsbok, och det kan göra att minnet tar slut i vissa konfigurationer, särskilt om du gör egna experiment som en del av samma anteckningsbok. Om du stöter på konstiga fel när du börjar träna modellen kan det vara bra att starta om anteckningsbokens kärna.


In [3]:
batch_size = 16
embed_size = 64

## Enkel RNN-klassificerare

I fallet med en enkel RNN är varje rekurrent enhet ett enkelt linjärt nätverk, som tar emot en inmatningsvektor och tillståndsvektor och producerar en ny tillståndsvektor. I Keras kan detta representeras av lagret `SimpleRNN`.

Även om vi kan skicka one-hot-kodade tokens direkt till RNN-lagret är detta inte en bra idé på grund av deras höga dimensioner. Därför kommer vi att använda ett inbäddningslager för att sänka dimensionerna av ordvektorer, följt av ett RNN-lager och slutligen en `Dense`-klassificerare.

> **Notera**: I fall där dimensionerna inte är så höga, till exempel vid teckenbaserad tokenisering, kan det vara vettigt att skicka one-hot-kodade tokens direkt in i RNN-cellen.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Notera:** Vi använder ett otränat inbäddningslager här för enkelhetens skull, men för bättre resultat kan vi använda ett förtränat inbäddningslager med Word2Vec, som beskrivs i föregående avsnitt. Det vore en bra övning för dig att anpassa denna kod för att fungera med förtränade inbäddningar.

Nu ska vi träna vår RNN. RNN:er är generellt ganska svåra att träna, eftersom när RNN-cellerna vecklas ut längs sekvensens längd blir det totala antalet lager som ingår i bakåtpropageringen ganska stort. Därför behöver vi välja en mindre inlärningshastighet och träna nätverket på en större dataset för att uppnå bra resultat. Detta kan ta ganska lång tid, så det är att föredra att använda en GPU.

För att snabba upp processen ska vi endast träna RNN-modellen på nyhetstitlar och utelämna beskrivningen. Du kan prova att träna med beskrivning och se om du kan få modellen att träna.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Observera** att noggrannheten troligtvis är lägre här, eftersom vi bara tränar på nyhetstitlar.


## Att återbesöka variabla sekvenser 

Kom ihåg att `TextVectorization`-lagret automatiskt fyller på sekvenser av variabel längd i en minibatch med pad-token. Det visar sig att dessa tokens också ingår i träningen, och de kan försvåra modellens konvergens.

Det finns flera tillvägagångssätt för att minimera mängden utfyllnad. En av dem är att omordna datasetet efter sekvenslängd och gruppera alla sekvenser efter storlek. Detta kan göras med hjälp av `tf.data.experimental.bucket_by_sequence_length`-funktionen (se [dokumentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Ett annat tillvägagångssätt är att använda **maskning**. I Keras stöder vissa lager extra inmatning som visar vilka token som ska beaktas vid träning. För att införliva maskning i vår modell kan vi antingen inkludera ett separat `Masking`-lager ([dokumentation](https://keras.io/api/layers/core_layers/masking/)), eller så kan vi specificera parametern `mask_zero=True` i vårt `Embedding`-lager.

> **Notera**: Denna träning tar cirka 5 minuter att genomföra en epok på hela datasetet. Känn dig fri att avbryta träningen när som helst om du tappar tålamodet. Vad du också kan göra är att begränsa mängden data som används för träning, genom att lägga till `.take(...)`-satsen efter `ds_train` och `ds_test` dataset.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Nu när vi använder maskering kan vi träna modellen på hela datasetet av titlar och beskrivningar.

> **Notera**: Har du märkt att vi har använt en vektorisator som tränats på nyhetstitlarna, och inte hela artikeltexten? Potentiellt kan detta göra att några av token ignoreras, så det är bättre att träna om vektorisatorn. Det kan dock bara ha en mycket liten effekt, så vi kommer att använda den tidigare förtränade vektorisatorn för enkelhetens skull.


## LSTM: Long short-term memory

Ett av huvudproblemen med RNN:er är **försvinnande gradienter**. RNN:er kan vara ganska långa, och kan ha svårt att propagera gradienterna hela vägen tillbaka till det första lagret i nätverket under bakåtriktad spridning. När detta händer kan nätverket inte lära sig relationer mellan avlägsna token. Ett sätt att undvika detta problem är att införa **explicit tillståndshantering** genom att använda **grindar**. De två vanligaste arkitekturerna som introducerar grindar är **long short-term memory** (LSTM) och **gated relay unit** (GRU). Vi kommer att behandla LSTM här.

![Bild som visar ett exempel på en long short term memory-celle](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

Ett LSTM-nätverk är organiserat på ett liknande sätt som en RNN, men det finns två tillstånd som förs från lager till lager: det faktiska tillståndet $c$, och den dolda vektorn $h$. Vid varje enhet kombineras den dolda vektorn $h_{t-1}$ med ingången $x_t$, och tillsammans styr de vad som händer med tillståndet $c_t$ och utgången $h_{t}$ genom **grindar**. Varje grind har sigmoidaktivering (utgång i intervallet $[0,1]$), vilket kan ses som en bitvis mask när den multipliceras med tillståndsvektorn. LSTM har följande grindar (från vänster till höger på bilden ovan):
* **glömskegrind** som bestämmer vilka komponenter i vektorn $c_{t-1}$ vi behöver glömma, och vilka som ska passera.
* **ingångsgrind** som bestämmer hur mycket information från ingångsvektorn och tidigare dold vektor som ska införlivas i tillståndsvektorn.
* **utgångsgrind** som tar den nya tillståndsvektorn och bestämmer vilka av dess komponenter som ska användas för att producera den nya dolda vektorn $h_t$.

Komponenterna i tillståndet $c$ kan ses som flaggor som kan slås på och av. Till exempel, när vi träffar på namnet *Alice* i sekvensen, antar vi att det refererar till en kvinna, och höjer flaggan i tillståndet som säger att vi har ett kvinnligt substantiv i meningen. När vi sedan träffar på orden *and Tom*, kommer vi att höja flaggan som säger att vi har ett flertal substantiv. Genom att manipulera tillståndet kan vi alltså hålla koll på de grammatiska egenskaperna i meningen.

> **Notera**: Här är en bra resurs för att förstå LSTM:erna inuti: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) av Christopher Olah.

Även om den interna strukturen av en LSTM-cell kan se komplex ut, döljer Keras denna implementering i `LSTM`-lagret, så det enda vi behöver göra i exemplet ovan är att byta ut det återkommande lagret:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Observera** att träning av LSTM:er också är ganska långsam, och du kanske inte ser någon större ökning av noggrannheten i början av träningen. Du kan behöva fortsätta träna ett tag för att uppnå bra noggrannhet.


## Dubbelsidiga och flerskiktade RNN:er

I våra exempel hittills har de rekurrenta nätverken arbetat från början av en sekvens till slutet. Detta känns naturligt för oss eftersom det följer samma riktning som vi läser eller lyssnar på tal. Men för scenarier som kräver slumpmässig åtkomst till inmatningssekvensen är det mer logiskt att köra den rekurrenta beräkningen i båda riktningarna. RNN:er som tillåter beräkningar i båda riktningar kallas **dubbelsidiga** RNN:er, och de kan skapas genom att omsluta det rekurrenta lagret med ett särskilt `Bidirectonal` lager.

> **Obs**: `Bidirectional`-lagret gör två kopior av lagret inuti sig, och sätter `go_backwards`-egenskapen för en av dessa kopior till `True`, vilket får det att gå i motsatt riktning längs sekvensen.

Rekurrenta nätverk, oavsett om de är enkelriktade eller dubbelsidiga, fångar mönster inom en sekvens och lagrar dem i tillståndsvektorer eller returnerar dem som output. Precis som med konvolutionsnätverk kan vi bygga ett annat rekurrent lager efter det första för att fånga högre nivåmönster, som byggs från lägre nivåmönster extraherade av det första lagret. Detta leder oss till begreppet **flerskikts-RNN**, som består av två eller flera rekurrenta nätverk, där output från föregående lager skickas till nästa lager som input.

![Bild som visar en Flerskikts long-short-term-memory- RNN](../../../../../translated_images/sv/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Bild från [detta fantastiska inlägg](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) av Fernando López.*

Keras gör konstruktionen av dessa nätverk enkel, eftersom du bara behöver lägga till fler rekurrenta lager i modellen. För alla lager utom det sista måste vi specificera parametern `return_sequences=True`, eftersom vi behöver att lagret returnerar alla mellanliggande tillstånd, inte bara det slutgiltiga tillståndet av den rekurrenta beräkningen.

Låt oss bygga en tvålagers dubbelsidig LSTM för vårt klassificeringsproblem.

> **Obs** denna kod tar återigen ganska lång tid att slutföra, men den ger oss den högsta noggrannheten vi sett hittills. Så kanske är det värt att vänta och se resultatet.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNN:er för andra uppgifter

Hittills har vi fokuserat på att använda RNN:er för att klassificera textsekvenser. Men de kan hantera många fler uppgifter, såsom textgenerering och maskinöversättning &mdash; vi kommer att behandla dessa uppgifter i nästa avsnitt.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
